Metadata Scraping
RAWG API = 164006f955cc4991b72b22a15d025e90

In [7]:
# ==============================================================================
# CELL 1: SETUP AND IMPORTS
# ==============================================================================
import pandas as pd
import requests
import time
from tqdm import tqdm
import numpy as np
import os

print("--- Step 1: Setup Complete ---")

# --- CONFIGURATION ---
# IMPORTANT: Replace with your actual RAWG API key
API_KEY = "164006f955cc4991b72b22a15d025e90"
INPUT_FILE = 'masterlist03.xlsx'
OUTPUT_FILE = 'masterlist_enriched03.csv'

if API_KEY == "YOUR_RAWG_API_KEY":
    print("🛑 HEY! Please replace 'YOUR_RAWG_API_KEY' with your actual key from rawg.io.")
else:
    print("✅ API Key loaded.")

--- Step 1: Setup Complete ---
✅ API Key loaded.


In [8]:
# ==============================================================================
# CELL 2: LOAD DATA AND FIX COLUMN NAMES
# ==============================================================================
try:
    df = pd.read_excel(INPUT_FILE)
    print("\n--- Step 2: Data Loaded Successfully ---")
    print("Original column names:", df.columns.tolist())

    df.rename(columns={
        'Game': 'game_name',
        'Publisher': 'publisher',
        'Developer': 'developer',
        'Release Date': 'release_date',
        'Metacritic': 'metacritic_score'
    }, inplace=True)

    # Ensure all target columns exist, creating them if they don't.
    for col in ['publisher', 'developer', 'release_date', 'metacritic_score']:
        if col not in df.columns:
            df[col] = np.nan
            
    print("✅ Corrected column names:", df.columns.tolist())

except FileNotFoundError:
    print(f"❌ ERROR: Could not find '{INPUT_FILE}'. Make sure it's in the same directory.")
    df = None


--- Step 2: Data Loaded Successfully ---
Original column names: ['Game', 'Release Date', 'Added to Service', 'Removed from Service', 'Metacritic', 'Publisher', 'Developer', 'System']
✅ Corrected column names: ['game_name', 'release_date', 'Added to Service', 'Removed from Service', 'metacritic_score', 'publisher', 'developer', 'System']


In [9]:
# ==============================================================================
# CELL 3: THE RELIABLE API FUNCTION
# ==============================================================================
def get_game_details(game_name, api_key):
    if not game_name or pd.isna(game_name):
        return None

    try:
        search_query = requests.utils.quote(str(game_name))
        search_url = f"https://api.rawg.io/api/games?key={api_key}&search={search_query}&page_size=1"
        response = requests.get(search_url)
        response.raise_for_status()
        data = response.json()
        game_slug = data['results'][0]['slug'] if data['results'] else None
    except requests.exceptions.RequestException:
        return None

    if not game_slug:
        return None

    try:
        details_url = f"https://api.rawg.io/api/games/{game_slug}?key={api_key}"
        response = requests.get(details_url)
        response.raise_for_status()
        game_data = response.json()

        publishers = [p['name'] for p in game_data.get('publishers', [])]
        developers = [d['name'] for d in game_data.get('developers', [])]
        
        # 🎯 ADDITION 1: Get the Metacritic score from the API response
        metacritic = game_data.get('metacritic', None)

        return {
            'publisher': ', '.join(publishers) if publishers else None,
            'developer': ', '.join(developers) if developers else None,
            'release_date': game_data.get('released', None),
            'metacritic_score': metacritic
        }
    except requests.exceptions.RequestException:
        return None

print("\n--- Step 3: API Helper Function Defined (with Metacritic logic) ---")


--- Step 3: API Helper Function Defined (with Metacritic logic) ---


In [10]:
# ==============================================================================
# NEW CELL: Pre-flight Test for the API Function
# ==============================================================================
print("\n--- Running a live test on the get_game_details function ---")

# Use a well-known game that is guaranteed to have full data
test_game_name = "The Witcher 3: Wild Hunt"
print(f"Attempting to fetch details for: '{test_game_name}'...")

# Make sure you have a valid API_KEY set in the first cell
if 'API_KEY' in locals() and API_KEY != "YOUR_RAWG_API_KEY":
    # Call the function just once for the test
    test_result = get_game_details(test_game_name, API_KEY)

    print("\nAPI Response:")
    print(test_result)

    # Check if all required fields were returned successfully
    if test_result and test_result.get('publisher') and test_result.get('developer') and test_result.get('metacritic_score'):
        print("\n✅ SUCCESS: The function correctly returned the publisher, developer, and metacritic score.")
        print("You are ready to proceed to the main data enrichment loop.")
    else:
        print("\n❌ WARNING: The function failed to retrieve all required data.")
        print("Please check your API key and the function code in the previous cell before proceeding.")
else:
    print("\n🛑 ERROR: API_KEY not set. Please set it in the first cell before running this test.")


--- Running a live test on the get_game_details function ---
Attempting to fetch details for: 'The Witcher 3: Wild Hunt'...

API Response:
{'publisher': 'CD PROJEKT RED', 'developer': 'CD PROJEKT RED', 'release_date': '2015-05-18', 'metacritic_score': 92}

✅ SUCCESS: The function correctly returned the publisher, developer, and metacritic score.
You are ready to proceed to the main data enrichment loop.


In [11]:
# ==============================================================================
# CELL 4: THE MAIN LOOP TO FETCH AND FILL DATA
# ==============================================================================
if df is not None:
    print("\n--- Step 4: Starting Data Enrichment Process ---")
    game_cache = {}
    pbar = tqdm(df.iterrows(), total=df.shape[0], desc="Enriching Game Data")

    for index, row in pbar:
        game_name = str(row.get('game_name', '')).strip()

        pub_missing = pd.isna(row.get('publisher')) or str(row.get('publisher', '')).strip() in ['', 'nan']
        dev_missing = pd.isna(row.get('developer')) or str(row.get('developer', '')).strip() in ['', 'nan']
        date_missing = pd.isna(row.get('release_date')) or str(row.get('release_date', '')).strip() in ['', 'nan']
        # 🎯 ADDITION 2: Check if metacritic_score is also missing
        meta_missing = pd.isna(row.get('metacritic_score')) or str(row.get('metacritic_score', '')).strip() in ['', 'nan']

        # The script will now run if ANY of these four are missing
        needs_fetching = pub_missing or dev_missing or date_missing or meta_missing

        if needs_fetching and game_name:
            pbar.set_postfix_str(f"Processing: {game_name}")

            if game_name in game_cache:
                details = game_cache[game_name]
            else:
                details = get_game_details(game_name, API_KEY)
                game_cache[game_name] = details
                time.sleep(0.5)

            if details:
                if pub_missing: df.loc[index, 'publisher'] = details.get('publisher')
                if dev_missing: df.loc[index, 'developer'] = details.get('developer')
                if date_missing: df.loc[index, 'release_date'] = details.get('release_date')
                # 🎯 ADDITION 3: Update the metacritic_score column if it was missing
                if meta_missing: df.loc[index, 'metacritic_score'] = details.get('metacritic_score')
        else:
            pbar.set_postfix_str(f"Skipping: {game_name}")
            
    print("\n✅ Data enrichment complete!")


--- Step 4: Starting Data Enrichment Process ---


Enriching Game Data:   0%|          | 0/894 [00:00<?, ?it/s, Processing: Grand Theft Auto III]C:\Users\Lyndon\AppData\Local\Temp\ipykernel_41916\2075598226.py:32: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Capcom, Rockstar Games, 1C-SoftClub, Buka Entertainment' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  if pub_missing: df.loc[index, 'publisher'] = details.get('publisher')
C:\Users\Lyndon\AppData\Local\Temp\ipykernel_41916\2075598226.py:33: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'DMA Design' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  if dev_missing: df.loc[index, 'developer'] = details.get('developer')
Enriching Game Data: 100%|██████████| 894/894 [34:10<00:00,  2.29s/it, Processing: Shadow Warrior 3]                   


✅ Data enrichment complete!


In [14]:
# ==============================================================================
# CELL 5: SAVE THE FINAL RESULT
# ==============================================================================
if df is not None:
    # Let's clean up the score to be a proper number, handling cases where it's not found
    df['metacritic_score'] = pd.to_numeric(df['metacritic_score'], errors='coerce')
    df.to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')
    
    print(f"\n--- Step 5: sSuccess! ---")
    print(f"🎉 Your enriched masterlist has been saved to '{OUTPUT_FILE}'.")
    print("\nFinal Data Preview:")
    display(df.head(10))


--- Step 5: sSuccess! ---
🎉 Your enriched masterlist has been saved to 'masterlist_enriched03.csv'.

Final Data Preview:


,game_name,release_date,Added to Service,Removed from Service,metacritic_score,publisher,developer,System
0,Grand Theft Auto III,2001-10-27,2025-06-10,NaT,97.0,"Capcom, Rockstar Games, 1C-SoftClub, Buka Ente...",DMA Design,PS5/PS4
1,God of War: Ragnarök,2022-11-09,2025-01-21,NaT,94.0,"Sony Interactive Entertainment, PlayStation PC",SCE Santa Monica Studio,PS5/PS4
2,Celeste,2018-01-25,2022-06-13,NaT,93.0,Matt Makes Games,"Matt Makes Games, Extremely OK Games, Noel",PS4
3,Blue Prince,2025-04-10,2025-04-10,NaT,92.0,Raw Fury,Dogubomb,PS5
4,Undertale,2017-08-15,2023-07-18,NaT,92.0,"8-4, Toby Fox","8-4, Toby Fox",PS4
5,Demon's Souls,2020-11-12,2022-06-13,NaT,92.0,Sony Computer Entertainment,"FromSoftware, Sony Computer Entertainment",PS5
6,Uncharted 4: A Thief's End,2016-05-01,2022-06-13,NaT,92.0,Sony Computer Entertainment,Naughty Dog,PS4
7,Crusader Kings III,2020-09-01,2024-06-18,NaT,91.0,Paradox Interactive,Paradox Developement Studios,PS5
8,Bloodborne,2015-09-01,2022-06-13,NaT,91.0,"Sony Computer Entertainment, Sony Interactive ...",FromSoftware,PS4
9,Hollow Knight: Voidheart Edition,2017-02-24,2022-06-13,NaT,91.0,Team Cherry,Team Cherry,PS4
